# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema, accessible here:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

*All dataset elements (record sets, fields, columns, etc.) are referenced using their `@id`s.*

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. The Croissant schema URL is used to initialize the Dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print metadata overview
print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We'll enumerate these to help target data extraction and analysis.

**Note:** All references must be by `@id`.

In [ ]:
# List all record sets defined in the dataset metadata
record_sets = metadata.recordSet
record_set_ids = []
print("Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    record_set_ids.append(rs['@id'])

# For each record set, show its fields
for rs in record_sets:
    rsid = rs['@id']
    print(f"\nFields in Record Set @id: {rsid}")
    if 'field' in rs and rs['field']:
        for f in rs['field']:
            print(f"  - @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
    else:
        print("  No fields available.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s identified above.

We'll collect all record sets, and load each set into pandas DataFrames using `mlcroissant.records(record_set=...)`.

In [ ]:
# Extract data for each record set
dataframes = {}

for record_set_id in record_set_ids:
    # Load records for this record set using its @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns in Record Set @id {record_set_id}:")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing numeric fields, grouping by key attributes.

We'll choose a numeric field (such as GAD-7 score) for analysis, referencing by `@id`. Adjust the threshold and group key as needed.

In [ ]:
# Example: EDA on main survey record set
# Select the primary record set that contains survey responses. Adjust as needed based on record_set_ids and previous overview.
primary_record_set_id = record_set_ids[0]
df = dataframes[primary_record_set_id]

# Identify a numeric field (e.g., GAD-7 score, PHQ-9 score, PSQ score) by @id
# Replace with the actual @id for GAD-7 score from the previous overview
numeric_field_id = None
group_field_id = None

# Search for numeric fields and a suitable group field
for col in df.columns:
    if 'gad7' in col.lower():
        numeric_field_id = col
    if 'village' in col.lower() or 'education' in col.lower():
        group_field_id = col

if numeric_field_id:
    # Filter: GAD-7 scores greater than a threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the GAD-7 scores (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field (e.g., village)
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize numeric score distributions and demographic grouping.

We'll use matplotlib and seaborn to illustrate GAD-7 scores, and optionally display group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 scores
if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} Scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Display grouped means
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
We have loaded and explored the FAIR² Kilifi County Mental Health Survey Dataset using `mlcroissant`.

Key steps included:
- Listing record sets, fields, and referencing them by `@id`.
- Extracting records and constructing DataFrames.
- Filtering, normalizing, and grouping numeric survey scores.
- Visualizing score distributions and group means.

This notebook template may be adapted to analyze other survey data leveraging Croissant data standards and the powerful utilities of `mlcroissant`.